# Phase 11 FixMatch pseudo-label audit (standalone, no training)

This notebook is **separate from the Phase 11 training notebook**. It loads the saved best FixMatch checkpoints and audits predictions on the unlabeled portion of each training split.

**It does not train or update weights. It is configured for CPU and does not require CUDA.** CPU inference can take a while.

Audit outputs include:
- class-wise acceptance counts/rates at the Phase 11 threshold (0.95)
- accepted pseudo-label precision, using manifest labels for a strictly post-hoc diagnostic
- weak/strong view class agreement
- pseudo-label class distribution

This is a diagnostic on the training pool, not a new test evaluation. Do not use its hidden-from-training labels for training or method selection. The strong augmentation is stochastic, so results describe the sampled strong views generated during this audit, not necessarily the exact views used in training.

## 1. Configure the project paths

The notebook attempts to locate your repository at `~/Documents/cervical-semi-sl` or from the current working directory. If it cannot find the split manifest, set `ROOT` manually.

Expected original Phase 11 output folder:
`data/results/phase11_controlled_v1/`

Expected run names:
`cric_5pct_fixmatch_convnext_tiny_seed42`, etc.

In [1]:
from pathlib import Path
import json, os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# CPU only: no CUDA is needed for this audit
DEVICE = torch.device("cpu")
BATCH_SIZE = 32
NUM_WORKERS = 0
PSEUDO_THRESHOLD = 0.95
SEED = 42
DATASETS = ("cric", "riva")
FRACTIONS = (0.05, 0.10)

_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.home() / "Documents" / "cervical-semi-sl",
    Path.home() / "OneDrive" / "Documents" / "cervical-semi-sl",
]
ROOT = next((p.resolve() for p in _candidates if (p / "data" / "splits_v2" / "cric_splits_v2.csv").exists()), None)

# If auto-detection fails, uncomment and set your repository folder:
# ROOT = Path(r"C:\Users\nanda\Documents\cervical-semi-sl")

if ROOT is None or not (ROOT / "data" / "splits_v2" / "cric_splits_v2.csv").exists():
    raise FileNotFoundError(
        "Could not locate data/splits_v2/cric_splits_v2.csv. "
        "Set ROOT to your cervical-semi-sl repository folder."
    )

SPLIT_DIR = ROOT / "data" / "splits_v2"
OUT_DIR = ROOT / "data" / "results" / "phase11_controlled_v1"
AUDIT_DIR = OUT_DIR / "fixmatch_audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print("Repository:", ROOT)
print("Phase 11 results:", OUT_DIR)
print("Audit outputs:", AUDIT_DIR)
print("Device:", DEVICE)


Repository: C:\Users\nanda\Documents\cervical-semi-sl
Phase 11 results: C:\Users\nanda\Documents\cervical-semi-sl\data\results\phase11_controlled_v1
Audit outputs: C:\Users\nanda\Documents\cervical-semi-sl\data\results\phase11_controlled_v1\fixmatch_audit
Device: cpu


## 2. Load manifests and recreate the exact unlabeled pool

In [2]:
def resolve_crop(path):
    p = Path(str(path))
    candidates = [p] if p.is_absolute() else [ROOT / p, ROOT / "data" / p]
    for c in candidates:
        if c.exists():
            return c.resolve()
    return None

manifests = {}
for ds in DATASETS:
    csv_path = SPLIT_DIR / f"{ds}_splits_v2.csv"
    if not csv_path.exists():
        raise FileNotFoundError(csv_path)
    df = pd.read_csv(csv_path)
    required = {"crop_path", "binary_label", "split"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{ds.upper()} manifest missing {missing}")
    df = df.copy()
    df["split"] = df["split"].astype(str).str.strip().str.lower().replace(
        {"validation":"val", "valid":"val"}
    )
    df["binary_label"] = pd.to_numeric(df["binary_label"], errors="raise").astype(int)
    df["resolved_path"] = df["crop_path"].map(resolve_crop)
    if df["resolved_path"].isna().any():
        raise FileNotFoundError(f"{ds.upper()}: some crop paths cannot be resolved")
    manifests[ds] = df.reset_index(drop=True)
    print(ds.upper(), "rows:", len(df), "| splits:", df["split"].value_counts().to_dict())

def get_labeled_and_unlabeled(ds, frac):
    train = manifests[ds].query("split == 'train'").reset_index(drop=True)
    subset_path = OUT_DIR / "label_subsets" / f"{ds}_labeled_{int(frac*100)}pct_seed{SEED}.csv"
    if not subset_path.exists():
        raise FileNotFoundError(
            f"Missing saved labeled subset: {subset_path}. "
            "The audit must use the exact subset from the original run."
        )
    saved = pd.read_csv(subset_path)
    if "resolved_path" not in saved:
        raise ValueError(f"{subset_path} does not contain resolved_path")
    selected = set(saved["resolved_path"].astype(str))
    labeled = train[train["resolved_path"].astype(str).isin(selected)].copy()
    if len(labeled) != len(selected):
        raise ValueError(f"Saved labeled subset does not match {ds.upper()} manifest.")
    unlabeled = train[~train["resolved_path"].astype(str).isin(selected)].copy()
    return labeled.reset_index(drop=True), unlabeled.reset_index(drop=True)

for ds in DATASETS:
    for frac in FRACTIONS:
        lab, unlab = get_labeled_and_unlabeled(ds, frac)
        print(f"{ds.upper()} {int(frac*100)}%: labeled={len(lab):,}; audit pool={len(unlab):,}")


CRIC rows: 11534 | splits: {'train': 8074, 'test': 2307, 'val': 1153}
RIVA rows: 15949 | splits: {'train': 11171, 'test': 3196, 'val': 1582}
CRIC 5%: labeled=404; audit pool=7,670
CRIC 10%: labeled=807; audit pool=7,267
RIVA 5%: labeled=559; audit pool=10,612
RIVA 10%: labeled=1,117; audit pool=10,054


## 3. Define the Phase 11 model and views

The original Phase 11 notebook used ConvNeXt-Tiny with a 2-class head, ImageNet normalization, weak view = resize + normalize, and strong view = flips + rotation + ColorJitter. This audit recreates those transforms. The strong view is randomly sampled at audit time.

In [3]:
IMAGE_SIZE = 128
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

weak_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
strong_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=.15, contrast=.15, saturation=.10, hue=.02),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class AuditDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        with Image.open(row["resolved_path"]) as im:
            im = im.convert("RGB")
            weak = weak_tf(im)
            strong = strong_tf(im)
        return weak, strong, int(row["binary_label"]), str(row["resolved_path"])

def build_model():
    # weights=None avoids downloading pretrained weights; checkpoint supplies trained weights.
    model = models.convnext_tiny(weights=None)
    model.classifier[2] = nn.Linear(model.classifier[2].in_features, 2)
    return model

def find_checkpoint(ds, frac):
    run_id = f"{ds}_{int(frac*100)}pct_fixmatch_convnext_tiny_seed{SEED}"
    checkpoint = OUT_DIR / run_id / "checkpoint_best.pt"
    if not checkpoint.exists():
        raise FileNotFoundError(f"Best checkpoint not found: {checkpoint}")
    return checkpoint, run_id


## 4. Run prediction-only audit and save per-example outputs

In [4]:
@torch.inference_mode()
def audit_checkpoint(ds, frac):
    labeled, unlabeled = get_labeled_and_unlabeled(ds, frac)
    checkpoint_path, run_id = find_checkpoint(ds, frac)
    model = build_model().to(DEVICE)
    payload = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    state = payload["model"] if isinstance(payload, dict) and "model" in payload else payload
    model.load_state_dict(state)
    model.eval()

    dl = DataLoader(
        AuditDataset(unlabeled), batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=False
    )
    records = []
    for weak, strong, true_label, paths in dl:
        weak_logits = model(weak.to(DEVICE))
        strong_logits = model(strong.to(DEVICE))
        weak_prob = torch.softmax(weak_logits.float(), dim=1).cpu().numpy()
        strong_prob = torch.softmax(strong_logits.float(), dim=1).cpu().numpy()
        pseudo = weak_prob.argmax(axis=1)
        confidence = weak_prob.max(axis=1)
        accepted = confidence >= PSEUDO_THRESHOLD
        strong_pred = strong_prob.argmax(axis=1)
        true_np = true_label.numpy()

        for i, path in enumerate(paths):
            records.append({
                "dataset": ds,
                "label_fraction": frac,
                "run_id": run_id,
                "image_path": path,
                "audit_true_label": int(true_np[i]),
                "weak_prob_0": float(weak_prob[i,0]),
                "weak_prob_1": float(weak_prob[i,1]),
                "weak_prediction": int(pseudo[i]),
                "weak_confidence": float(confidence[i]),
                "accepted": bool(accepted[i]),
                "strong_prob_0": float(strong_prob[i,0]),
                "strong_prob_1": float(strong_prob[i,1]),
                "strong_prediction": int(strong_pred[i]),
                "weak_strong_agree": bool(pseudo[i] == strong_pred[i]),
                "pseudo_correct_posthoc": bool(pseudo[i] == true_np[i]),
            })

    result = pd.DataFrame(records)
    out_csv = AUDIT_DIR / f"{ds}_{int(frac*100)}pct_fixmatch_pseudo_audit.csv"
    result.to_csv(out_csv, index=False)
    print(f"Saved {len(result):,} rows -> {out_csv}")
    del model
    return result

audit_frames = {}
for dataset in DATASETS:
    for fraction in FRACTIONS:
        key = f"{dataset}_{int(fraction*100)}pct"
        audit_frames[key] = audit_checkpoint(dataset, fraction)


Saved 7,670 rows -> C:\Users\nanda\Documents\cervical-semi-sl\data\results\phase11_controlled_v1\fixmatch_audit\cric_5pct_fixmatch_pseudo_audit.csv
Saved 7,267 rows -> C:\Users\nanda\Documents\cervical-semi-sl\data\results\phase11_controlled_v1\fixmatch_audit\cric_10pct_fixmatch_pseudo_audit.csv
Saved 10,612 rows -> C:\Users\nanda\Documents\cervical-semi-sl\data\results\phase11_controlled_v1\fixmatch_audit\riva_5pct_fixmatch_pseudo_audit.csv
Saved 10,054 rows -> C:\Users\nanda\Documents\cervical-semi-sl\data\results\phase11_controlled_v1\fixmatch_audit\riva_10pct_fixmatch_pseudo_audit.csv


## 5. Summaries: acceptance rates, accepted-label precision, agreement, and class distribution

In [5]:
def summarize_audit(df):
    # Pseudo class 0 = Normal; 1 = Abnormal
    x = df.copy()
    x["pseudo_class_name"] = x["weak_prediction"].map({0:"Normal", 1:"Abnormal"})
    by_class = x.groupby("pseudo_class_name", dropna=False).agg(
        pool_count=("weak_prediction", "size"),
        accepted_count=("accepted", "sum"),
        acceptance_rate=("accepted", "mean"),
        accepted_precision_posthoc=("pseudo_correct_posthoc", lambda s: float(s[x.loc[s.index, "accepted"]].mean())
                                    if x.loc[s.index, "accepted"].any() else np.nan),
        weak_strong_agreement=("weak_strong_agree", "mean"),
    )
    return by_class

summary_rows = []
for key, df in audit_frames.items():
    print("\\n###", key)
    display(summarize_audit(df))
    summary_rows.append({
        "run": key,
        "n_unlabeled": len(df),
        "acceptance_rate": float(df["accepted"].mean()),
        "weak_strong_agreement": float(df["weak_strong_agree"].mean()),
        "accepted_precision_posthoc": float(df.loc[df["accepted"], "pseudo_correct_posthoc"].mean())
            if df["accepted"].any() else np.nan,
        "pseudo_normal_fraction": float((df["weak_prediction"] == 0).mean()),
        "pseudo_abnormal_fraction": float((df["weak_prediction"] == 1).mean()),
        "accepted_normal_count": int(((df["weak_prediction"] == 0) & df["accepted"]).sum()),
        "accepted_abnormal_count": int(((df["weak_prediction"] == 1) & df["accepted"]).sum()),
    })

summary_df = pd.DataFrame(summary_rows)
summary_path = AUDIT_DIR / "fixmatch_audit_summary.csv"
summary_df.to_csv(summary_path, index=False)
display(summary_df)
print("Summary saved:", summary_path)


\n### cric_5pct


,pool_count,accepted_count,acceptance_rate,accepted_precision_posthoc,weak_strong_agreement
pseudo_class_name,,,,,
Abnormal,3909,3444,0.881044,0.877468,0.947045
Normal,3761,3262,0.867323,0.966891,0.961181


\n### cric_10pct


,pool_count,accepted_count,acceptance_rate,accepted_precision_posthoc,weak_strong_agreement
pseudo_class_name,,,,,
Abnormal,3498,2394,0.684391,0.962406,0.944826
Normal,3769,2546,0.675511,0.986253,0.961528


\n### riva_5pct


,pool_count,accepted_count,acceptance_rate,accepted_precision_posthoc,weak_strong_agreement
pseudo_class_name,,,,,
Abnormal,3575,2661,0.744336,0.803833,0.898462
Normal,7037,5694,0.809152,0.897085,0.950263


\n### riva_10pct


,pool_count,accepted_count,acceptance_rate,accepted_precision_posthoc,weak_strong_agreement
pseudo_class_name,,,,,
Abnormal,3562,1079,0.302920,0.876738,0.924761
Normal,6492,4169,0.642175,0.931159,0.937770


,run,n_unlabeled,acceptance_rate,weak_strong_agreement,accepted_precision_posthoc,pseudo_normal_fraction,pseudo_abnormal_fraction,accepted_normal_count,accepted_abnormal_count
0,cric_5pct,7670,0.874316,0.953977,0.920966,0.490352,0.509648,3262,3444
1,cric_10pct,7267,0.679785,0.953488,0.974696,0.518646,0.481354,2546,2394
2,riva_5pct,10612,0.787316,0.932812,0.867385,0.663117,0.336883,5694,2661
3,riva_10pct,10054,0.521981,0.933161,0.919970,0.645713,0.354287,4169,1079


Summary saved: C:\Users\nanda\Documents\cervical-semi-sl\data\results\phase11_controlled_v1\fixmatch_audit\fixmatch_audit_summary.csv


## Interpretation guardrails

- Class-wise acceptance disparity is a diagnostic signal, not proof that the minority class is being unfairly rejected.
- Accepted pseudo-label precision is computed against the manifest's labels after prediction. These labels were not used by the model, but because they belong to the training pool, use the result only for diagnosis, not for selecting/tuning a method.
- Weak/strong disagreement indicates view sensitivity. It does not by itself prove that augmentation destroys diagnostic morphology.
- Pseudo-label distribution skew does not establish true class-prior mismatch.
- This notebook audits the unlabeled training pool. It does not touch validation/test sets, and it does not perform training or CUDA operations.